# Hybrid Recommendation Engine

This notebook combines all components for the final recommendation system:
1. GNN embeddings (from Phase 4)
2. DCI Closed itemset mining
3. XGBoost ranking with gBCE calibration
4. Comprehensive evaluation

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

from src.config import get_config, PROCESSED_DATA_DIR, MODEL_DIR
from src.models.hybrid import (
    HybridRecommender,
    UserProfileMiner,
    DCIClosed,
    CalibrationMetrics,
    calibrate_predictions,
)
from src.utils.helpers import set_seed
from src.utils.metrics import RecommendationMetrics

# Setup
set_seed(42)
config = get_config()

## 1. Load Data and Embeddings

In [ ]:
# Load interactions
train_reviews = pd.read_parquet(PROCESSED_DATA_DIR / 'train_reviews.parquet')
val_reviews = pd.read_parquet(PROCESSED_DATA_DIR / 'val_reviews.parquet')
test_reviews = pd.read_parquet(PROCESSED_DATA_DIR / 'test_reviews.parquet')

print(f"Train: {len(train_reviews)}")
print(f"Val: {len(val_reviews)}")
print(f"Test: {len(test_reviews)}")

In [ ]:
# Load GNN embeddings
gnn_dir = MODEL_DIR / 'gnn'

if (gnn_dir / 'user_gnn_embeddings.npy').exists():
    user_embeddings = np.load(gnn_dir / 'user_gnn_embeddings.npy')
    venue_embeddings = np.load(gnn_dir / 'venue_gnn_embeddings.npy')
    with open(gnn_dir / 'id_mappings.json') as f:
        id_mappings = json.load(f)
    user_ids = id_mappings['user_ids']
    venue_ids = id_mappings['venue_ids']
    print(f"Loaded GNN embeddings")
    print(f"  Users: {user_embeddings.shape}")
    print(f"  Venues: {venue_embeddings.shape}")
else:
    print("GNN embeddings not found, creating random initialization")
    user_ids = train_reviews['user_id'].unique().tolist()
    venue_ids = train_reviews['business_id'].unique().tolist()
    user_embeddings = np.random.randn(len(user_ids), 64).astype(np.float32) * 0.1
    venue_embeddings = np.random.randn(len(venue_ids), 64).astype(np.float32) * 0.1

In [ ]:
# Create ID mappings
user_id_map = {uid: idx for idx, uid in enumerate(user_ids)}
venue_id_map = {vid: idx for idx, vid in enumerate(venue_ids)}

print(f"Users in mapping: {len(user_id_map)}")
print(f"Venues in mapping: {len(venue_id_map)}")

## 2. DCI Closed Itemset Mining

In [ ]:
# Mine user behavior patterns
miner = UserProfileMiner(
    min_support=0.01,
    max_itemset_size=5,
)

miner.fit(train_reviews, user_col='user_id', venue_col='business_id')

print(f"Mined closed itemsets: {len(miner.dci.closed_itemsets)}")

In [ ]:
# Show top patterns
top_patterns = sorted(
    miner.dci.closed_itemsets,
    key=lambda x: (-x.support, -len(x.items))
)[:10]

print("Top 10 Closed Itemsets:")
for i, pattern in enumerate(top_patterns, 1):
    print(f"  {i}. Support: {pattern.support:.4f}, Size: {len(pattern.items)}, Count: {pattern.count}")

In [ ]:
# Get user profiles with itemset features
ITEMSET_FEATURES = 10
user_profiles_df = miner.get_all_profiles_df(itemset_features=ITEMSET_FEATURES)

print(user_profiles_df.head())
print(f"\nProfile columns: {user_profiles_df.columns.tolist()}")

In [ ]:
# Create user extra features matrix
itemset_cols = [f'itemset_{i}' for i in range(ITEMSET_FEATURES)]
user_extra = np.zeros((len(user_ids), ITEMSET_FEATURES), dtype=np.float32)

for _, row in user_profiles_df.iterrows():
    uid = row['user_id']
    if uid in user_id_map:
        idx = user_id_map[uid]
        for i, col in enumerate(itemset_cols):
            user_extra[idx, i] = row[col]

print(f"User extra features shape: {user_extra.shape}")
print(f"Non-zero entries: {(user_extra != 0).sum()}")

## 3. Prepare Training Data

In [ ]:
# Convert reviews to edge arrays
def reviews_to_edges(reviews_df):
    users = []
    venues = []
    for _, row in reviews_df.iterrows():
        uid = row['user_id']
        vid = row['business_id']
        if uid in user_id_map and vid in venue_id_map:
            users.append(user_id_map[uid])
            venues.append(venue_id_map[vid])
    return np.array([users, venues])

train_edges = reviews_to_edges(train_reviews)
val_edges = reviews_to_edges(val_reviews)
test_edges = reviews_to_edges(test_reviews)

print(f"Train edges: {train_edges.shape}")
print(f"Val edges: {val_edges.shape}")
print(f"Test edges: {test_edges.shape}")

## 4. Train Hybrid Model

In [ ]:
# Configure
config.hybrid.xgb_n_estimators = 50  # Reduce for demo
config.hybrid.gbce_calibration_t = 0.8

# Create recommender
recommender = HybridRecommender(config=config.hybrid)

In [ ]:
# Train
history = recommender.fit(
    train_edges=train_edges,
    user_embeddings=user_embeddings,
    venue_embeddings=venue_embeddings,
    user_ids=user_ids,
    venue_ids=venue_ids,
    user_extra=user_extra,
    val_edges=val_edges,
    num_negatives=4,
    num_rounds=50,
)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# AUC
if 'train' in history and 'auc' in history['train']:
    axes[0].plot(history['train']['auc'], label='Train')
if 'val' in history and 'auc' in history['val']:
    axes[0].plot(history['val']['auc'], label='Val')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('AUC')
axes[0].set_title('Training AUC')
axes[0].legend()
axes[0].grid(True)

# Log Loss
if 'train' in history and 'logloss' in history['train']:
    axes[1].plot(history['train']['logloss'], label='Train')
if 'val' in history and 'logloss' in history['val']:
    axes[1].plot(history['val']['logloss'], label='Val')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Log Loss')
axes[1].set_title('Training Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 5. Evaluate Recommendations

In [ ]:
# Sample test users
test_user_sample = test_reviews['user_id'].unique()[:100]

# Get ground truth
user_ground_truth = {}
for uid in test_user_sample:
    venues = test_reviews[test_reviews['user_id'] == uid]['business_id'].tolist()
    user_ground_truth[uid] = set(venues)

# Get recommendations
user_recommendations = {}
for uid in test_user_sample:
    recs = recommender.recommend(uid, k=20, exclude_visited=True)
    user_recommendations[uid] = [vid for vid, _ in recs]

print(f"Generated recommendations for {len(user_recommendations)} users")

In [ ]:
# Compute metrics
metrics = RecommendationMetrics()
results = metrics.evaluate_all(
    user_recommendations,
    user_ground_truth,
    k_values=[5, 10, 20],
)

print("Test Set Metrics:")
for metric, value in sorted(results.items()):
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Visualize metrics
k_values = [5, 10, 20]
metric_names = ['precision', 'recall', 'ndcg', 'hit_rate']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, metric in enumerate(metric_names):
    values = [results[f'{metric}@{k}'] for k in k_values]
    axes[i].bar(k_values, values, color='steelblue')
    axes[i].set_xlabel('K')
    axes[i].set_ylabel(metric.capitalize())
    axes[i].set_title(f'{metric.upper()}@K')
    axes[i].set_xticks(k_values)

plt.tight_layout()
plt.show()

## 6. Calibration Analysis

In [ ]:
# Get predictions for calibration analysis
test_user_idx = test_edges[0]
test_venue_idx = test_edges[1]

# Positive predictions
pos_scores = recommender.ranker.predict(
    test_user_idx,
    test_venue_idx,
    user_embeddings,
    venue_embeddings,
    user_extra,
)

# Negative predictions
neg_venue_idx = np.random.randint(0, len(venue_ids), len(test_user_idx))
neg_scores = recommender.ranker.predict(
    test_user_idx,
    neg_venue_idx,
    user_embeddings,
    venue_embeddings,
    user_extra,
)

all_scores = np.concatenate([pos_scores, neg_scores])
all_labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])

print(f"Positive scores mean: {pos_scores.mean():.4f}")
print(f"Negative scores mean: {neg_scores.mean():.4f}")

In [ ]:
# Compute calibration metrics
ece = CalibrationMetrics.expected_calibration_error(all_scores, all_labels)
mce = CalibrationMetrics.maximum_calibration_error(all_scores, all_labels)

print(f"Expected Calibration Error (ECE): {ece:.4f}")
print(f"Maximum Calibration Error (MCE): {mce:.4f}")

In [ ]:
# Reliability diagram
bin_centers, bin_accuracies, bin_counts = CalibrationMetrics.reliability_diagram(
    all_scores, all_labels, n_bins=10
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Reliability diagram
axes[0].bar(bin_centers, bin_accuracies, width=0.08, alpha=0.7, label='Accuracy')
axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect calibration')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Actual Accuracy')
axes[0].set_title(f'Reliability Diagram (ECE={ece:.4f})')
axes[0].legend()
axes[0].grid(True)

# Score distribution
axes[1].hist(pos_scores, bins=30, alpha=0.5, label='Positive', density=True)
axes[1].hist(neg_scores, bins=30, alpha=0.5, label='Negative', density=True)
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
# Get feature importance
importance = recommender.ranker.get_feature_importance()

# Plot top features
top_n = 20
top_features = importance.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_features)), top_features['importance'].values)
plt.yticks(range(len(top_features)), top_features['feature'].values)
plt.xlabel('Importance (Gain)')
plt.title(f'Top {top_n} Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Sample Recommendations

In [ ]:
# Get recommendations for a specific user
sample_user_id = test_user_sample[0]

recs = recommender.recommend(sample_user_id, k=10, exclude_visited=True)

print(f"Top 10 recommendations for user {sample_user_id}:")
print(f"{'Rank':<6} {'Venue ID':<25} {'Score':<10}")
print("-" * 45)

for rank, (venue_id, score) in enumerate(recs, 1):
    print(f"{rank:<6} {venue_id:<25} {score:.4f}")

In [ ]:
# Show user's actual visits
user_visits = train_reviews[train_reviews['user_id'] == sample_user_id]
print(f"\nUser's training visits ({len(user_visits)} total):")
print(user_visits[['business_id', 'stars']].head(10))

## 9. Save Model

In [ ]:
# Save
output_dir = MODEL_DIR / 'hybrid'
output_dir.mkdir(parents=True, exist_ok=True)

recommender.ranker.save(output_dir / 'xgboost_ranker.json')
importance.to_csv(output_dir / 'feature_importance.csv', index=False)

# Save results
results_df = pd.DataFrame([results])
results_df.to_csv(output_dir / 'test_results.csv', index=False)

print(f"Saved model and results to {output_dir}")

## Summary

The hybrid recommendation system combines:
1. **GNN embeddings** - Capture user-venue interaction patterns
2. **DCI Closed itemsets** - Mine frequent behavior patterns
3. **XGBoost ranking** - Learn non-linear feature interactions
4. **gBCE calibration** - Reduce overconfidence in predictions

This provides personalized, calibrated venue recommendations!